# 第3课：混合检索与答案生成

## 本节目标

理解 RAG 的"检索"和"生成"两个核心环节。你将学会：

1. **BM25 关键词检索**：基于词频的精确匹配
2. **向量语义检索**：基于向量相似度的模糊匹配
3. **混合检索（Hybrid Retrieval）**：结合两者优势
4. **RRF 融合算法**：如何公平地合并两种排序结果
5. **LLM 答案生成**：设计 Prompt 模板，让模型基于检索结果回答

> 核心洞察：BM25 懂"词"，向量检索懂"意"。混合检索 = 既懂词又懂意。

## 环境准备

In [ ]:
import sys
sys.path.insert(0, '..')

from config import config
from src.pipeline import RAGPipeline

print("模块加载成功!")

## 1. 构建完整 RAG 流水线

`RAGPipeline` 封装了完整流程：Load → Chunk → Embed → Index → Retrieve → Generate。

首次运行需要下载嵌入模型（~400MB），请耐心等待。

In [ ]:
# 构建索引（自动完成加载→分块→嵌入→存储→BM25索引）
pipeline = RAGPipeline(
    strategy="recursive_char",   # 分块策略
    store_type="chroma",          # 向量存储
    embedding_model="BAAI/bge-small-zh-v1.5",
)
pipeline.build_index()
print("RAG 流水线就绪!")

## 2. 对比三种检索方式

同一个问题，三种检索方式会返回什么结果？这是理解混合检索价值的最佳方式。

In [ ]:
question = "Explain how gradient descent works"

# 分别使用三种方式检索
bm25_results = pipeline.retriever.retrieve_bm25_only(question)
vector_results = pipeline.retriever.retrieve_vector_only(question)
hybrid_results = pipeline.retriever.retrieve(question)

def show_results(name, results):
    print(f"\n{'='*60}")
    print(f"{name} (Top 3)")
    print(f"{'='*60}")
    for i, r in enumerate(results[:3], 1):
        print(f"  [{i}] 相关度={r['score']:.4f} | 来源={r['metadata']['source']}")
        print(f"      {r['content'][:120]}...")

show_results("BM25 关键词检索", bm25_results)
show_results("向量语义检索", vector_results)
show_results("混合检索 (RRF融合)", hybrid_results)

## 3. 理解 RRF（Reciprocal Rank Fusion）

混合检索的核心挑战：BM25 的分数和向量相似度在完全不同的尺度上，不能直接相加。

**RRF 的解决方案**：不看分数绝对值，看排名。

$$RRF\_score(d) = \sum_{i \in retrievers} \frac{w_i}{k + rank_i(d)}$$

- $k=60$ 是一个平滑常数，防止排名第1的文档权重过大
- $w_i$ 是每个检索器的权重（默认 BM25=0.2, Vector=0.8）

> 通俗理解：两场比赛（BM25赛和向量赛），不看具体成绩，看名次。综合名次最高的就是最好的。

In [ ]:
# BM25 和向量检索的结果重叠度
bm25_ids = set(r['id'] for r in bm25_results[:5])
vector_ids = set(r['id'] for r in vector_results[:5])
hybrid_ids = set(r['id'] for r in hybrid_results[:5])

print(f"BM25 Top-5 独有: {len(bm25_ids - vector_ids)} 个")
print(f"向量 Top-5 独有: {len(vector_ids - bm25_ids)} 个")
print(f"共同结果: {len(bm25_ids & vector_ids)} 个")
print(f"混合检索包含了 BM25 的 {len(hybrid_ids & bm25_ids)} 个 + 向量的 {len(hybrid_ids & vector_ids)} 个")

## 4. 仅检索模式（不调用 LLM）

有时我们只需要找到相关文档，不需要生成回答。这在调试和验证阶段非常有用。

In [ ]:
# use_llm=False 只检索不生成
result = pipeline.query("What is the bias-variance tradeoff?", use_llm=False)

print(f"问题: {result['query']}")
print(f"检索耗时: {result['retrieval_time_ms']}ms")
print(f"\n检索到的文档:")
for doc in result['retrieved_docs']:
    print(f"  [{doc['score']:.4f}] {doc['source']}")

## 5. LLM 答案生成

设置好 API Key 后，LLM 会基于检索到的文档生成结构化的回答。

### Prompt 模板设计

好的 Prompt 决定了生成质量：

```
你是一位知识渊博的机器学习助教。请根据提供的课程讲义内容回答用户的问题。
仅使用上下文中的信息作答。如果上下文不足以回答问题，请如实说明。

## 上下文（课程讲义）
{检索到的文档内容}

## 用户问题
{用户的问题}

## 回答要求
- 基于上述上下文内容作答
- 适当使用数学符号表达
- 回答简洁但全面
```

关键设计原则：

- **角色设定**：限定 LLM 的行为边界
- **仅使用上下文**：防止幻觉，强制引用检索结果
- **格式约束**：让输出结构化
- **诚实兜底**：不知道就说不知道

In [ ]:
# 要使用 LLM 生成回答，需要先配置 API Key：
# 1. 复制 .env.example 为 .env
# 2. 编辑 .env 填入你的 DEEPSEEK_API_KEY 或 OPENAI_API_KEY
# 3. 重启 notebook kernel 后取消下面的注释
#
# 配置完成后取消注释运行：
# result = pipeline.query("What is logistic regression?", use_llm=True)
# print(f"问题: {result['query']}")
# print(f"模型: {result.get('model', 'unknown')}")
# print(f"\n回答:\n{result['answer']}")
# print(f"\n引用来源: {result['sources']}")
# print(f"检索耗时: {result['retrieval_time_ms']}ms")

print("要使用 LLM 生成回答：")
print("  1. cp .env.example .env")
print("  2. 编辑 .env 填入 API Key")
print("  3. 取消上方代码注释并重新运行此 cell")

## 6. 完整流程验证

使用 `compare_retrieval_methods` 一次性对比所有检索方式。

In [ ]:
question = "What is cross-validation?"
comparison = pipeline.compare_retrieval_methods(question)

for method in ["bm25", "vector", "hybrid"]:
    name = {"bm25": "BM25关键词", "vector": "向量语义", "hybrid": "混合检索"}[method]
    print(f"\n{'='*60}")
    print(f"{name} 检索结果")
    print(f"{'='*60}")
    for i, doc in enumerate(comparison[method][:3], 1):
        print(f"  [{i}] {doc['source']} (相关度: {doc['score']:.4f})")
        print(f"      {doc['preview'][:100]}...")

## 本节小结

- BM25（精确）和向量检索（语义）互补，混合检索效果最佳
- RRF 是简单有效的融合算法，不需要归一化分数
- Prompt 模板设计是生成质量的关键
- 仅检索模式是调试和验证的好工具

**下一步**：在 04 号笔记本中，我们将系统评估 RAG 的各个组件，量化对比不同策略的效果。